# 🤖 Bemo Chatbot — Google Gemini API

> **Bemo** is a smart AI assistant powered by **Gemini 2.5 Flash**.  
> She can search the web, solve math, check the weather, tell the time, summarize files, and remember your conversation.

---

| Feature | Detail |
|---|---|
| 🧠 Model | `gemini-2.5-flash` |
| 🔎 Web Search | DuckDuckGo (ddgs) |
| 🧮 Calculator | Built-in popup |
| 🌤️ Weather | wttr.in |
| 🕐 Date & Time | Python `datetime` |
| 📄 File Summary | Upload any text file |
| 💬 Memory | Last 5 turns |
| 🖥️ Interface | Tkinter GUI |

---


## 📦 1 · Install Dependencies

In [1]:
!pip install -q google-generativeai duckduckgo-search sympy ddgs pyttsx3 SpeechRecognition pyaudio opencv-python Pillow

## 🔑 2 · API Key

Enter your **Google AI Studio** key when prompted.  
Get one free at [aistudio.google.com](https://aistudio.google.com/app/apikey).

In [2]:
import os
from getpass import getpass

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google API key: ")

## 🧠 3 · Load Model

In [3]:
import google.generativeai as genai
import os, time, re

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))
_raw_model = genai.GenerativeModel("gemini-2.5-flash")

class _ModelWrapper:
    """Wraps GenerativeModel — auto-retries on 429 with countdown."""

    def generate_content(self, prompt, **kwargs):
        max_attempts = 5
        wait         = 20   # seconds for first retry

        for attempt in range(1, max_attempts + 1):
            try:
                return _raw_model.generate_content(prompt, **kwargs)

            except Exception as e:
                msg = str(e)

                # parse suggested retry delay from error message
                match = re.search(r"retry[^\d]*(\d+)", msg, re.I)
                suggested = int(match.group(1)) + 2 if match else wait

                if "429" in msg or "quota" in msg.lower():
                    if attempt == max_attempts:
                        raise
                    # live countdown in status bar
                    for s in range(suggested, 0, -1):
                        try:
                            status_var.set(
                                f"⏳ Rate limit — retrying in {s}s "
                                f"(attempt {attempt}/{max_attempts - 1})…"
                            )
                            root.update_idletasks()
                        except Exception:
                            pass
                        time.sleep(1)
                    wait = min(wait * 2, 120)   # exponential back-off
                else:
                    raise   # not a quota error — re-raise immediately

model = _ModelWrapper()


C:\Users\Mohamed Mahmoud\AppData\Local\Temp\ipykernel_8036\2011898416.py:1: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## 🛠️ 4 · Tools

Each tool is a plain Python function.  
The agent calls the right one based on what the user asks.

In [4]:
# ── 🔎 Web Search ──────────────────────────────────────────────────────────
from ddgs import DDGS

def web_search(query):
    results = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=3):
            results.append(r['body'])
    return "\n".join(results)

In [5]:
# ── 🧮 Calculator ──────────────────────────────────────────────────────────
import sympy as sp

def calculator(query):
    try:
        return str(sp.sympify(query))
    except:
        return "Error in calculation"

In [6]:
# ── 🌤️ Weather ─────────────────────────────────────────────────────────────
import requests

def get_weather(city):
    url = f"https://wttr.in/{city}?format=3"
    try:
        return requests.get(url).text
    except:
        return "Weather unavailable"

In [7]:
# ── 🕐 Date & Time ─────────────────────────────────────────────────────────
from datetime import datetime

def get_datetime(_=None):
    now = datetime.now()
    return now.strftime("📅 %A, %d %B %Y  |  🕐 %I:%M %p")

In [8]:
# ── 💬 Memory ──────────────────────────────────────────────────────────────
memory = []

def save_memory(user, bot):
    memory.append({"user": user, "bot": bot})

def get_memory():
    """Return full conversation history formatted for the prompt."""
    if not memory:
        return "No previous conversation."
    lines = []
    for turn in memory:
        lines.append(f"User: {turn['user']}")
        lines.append(f"Bemo: {turn['bot']}")
    return "\n".join(lines)

def memory_count():
    return len(memory)


## 🔀 5 · Tool Router

Sends the user's message to Gemini and gets back a single keyword  
(`SEARCH`, `CALCULATE`, `WEATHER`, `DATETIME`, or `NONE`) to decide which tool to run.

In [9]:
# Keywords that trigger local tools (no extra API call needed for detection)
WEATHER_KW  = {"weather","طقس","جو","حرارة","درجة"}
CALC_KW     = set("0123456789+-*/^") | {"calc","حساب","احسب","كم","يساوي","="}
DATETIME_KW = {"time","date","وقت","تاريخ","النهارده","today","اليوم","الساعة"}

def fast_decide(text):
    """Rule-based pre-filter — zero API calls for obvious cases."""
    low = text.lower()
    words = set(low.split())
    if words & WEATHER_KW:                    return "WEATHER"
    if words & DATETIME_KW:                   return "DATETIME"
    # if input is mostly math symbols/digits
    math_chars = sum(1 for c in text if c in "0123456789+-*/().^ ")
    if math_chars / max(len(text), 1) > 0.6:  return "CALCULATE"
    return None   # unknown — needs API

TOOLS = {"SEARCH", "CALCULATE", "WEATHER", "DATETIME", "NONE"}

def decide_tool(user_input):
    """Try rule-based first; fall back to a very short Gemini call."""
    quick = fast_decide(user_input)
    if quick:
        return quick
    prompt = (
        "Reply with ONE word only — SEARCH, CALCULATE, WEATHER, DATETIME, or NONE.\n"
        f"Question: {user_input}"
    )
    res = model.generate_content(prompt)
    decision = res.text.strip().upper()
    for t in TOOLS:
        if t in decision:
            return t
    return "NONE"


## 🎭 6 · Bemo's Personality & Response Generator

In [10]:
PERSONALITY = """
You are Bemo 🤖 — a sharp, witty, and genuinely helpful AI assistant.

Your style:
- Warm and conversational, like a smart friend — not a formal assistant.
- You keep answers concise and clear; no unnecessary fluff.
- You add a touch of dry humor when it fits, but never force it.
- You speak the same language the user writes in (Arabic, English, etc.).
- If the user is confused, you re-explain in a simpler way without being condescending.
- You remember the full conversation and naturally reference earlier points when relevant.
- You never say "As an AI…" or "I cannot feel…" — just be real and helpful.
"""

def generate_response(user_input, tool, tool_result):
    history = get_memory()
    tool_section = f"Tool Result:\n{tool_result}" if tool_result else ""

    prompt = f"""
{PERSONALITY}

Full conversation so far:
{history}

User: {user_input}

{tool_section}

Answer naturally. Reference earlier conversation if relevant.
"""
    res = model.generate_content(prompt)
    return res.text


## 🤖 7 · Agent

Ties everything together: routes → runs tool → generates response → saves to memory.

In [11]:
import threading

TOOL_FN = {
    "SEARCH"   : web_search,
    "CALCULATE": calculator,
    "WEATHER"  : get_weather,
    "DATETIME" : get_datetime,
}

def agent(user_input):
    tool        = decide_tool(user_input)
    tool_result = TOOL_FN[tool](user_input) if tool in TOOL_FN else None
    response    = generate_response(user_input, tool, tool_result)
    save_memory(user_input, response)
    return response

# ── 📄 File Summariser ─────────────────────────────────────────────────────
def summarize_file(path):
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        prompt = (
            "You are Bemo. Summarize the following file clearly and concisely.\n"
            f"File:\n{content[:8000]}"
        )
        return model.generate_content(prompt).text
    except Exception as e:
        return f"Could not read file: {e}"


## 🖥️ 8 · Bemo GUI

Dark-themed chat window with:
- 💬 Chat with Bemo
- 📄 Upload a file to summarize (txt, py, md, csv …)
- 🧮 Pop-up calculator


## 🎙️ 9 · Voice Features

Two voice capabilities:
- **TTS** — Bemo speaks her replies using `pyttsx3` (offline)
- **STT** — 🎤 button lets you speak instead of type using `SpeechRecognition`

In [12]:
import pyttsx3
import speech_recognition as sr
import threading

# ── TTS Engine ─────────────────────────────────────────────────────────────
_tts_engine = pyttsx3.init()
_tts_engine.setProperty("rate", 165)      # speaking speed
_tts_engine.setProperty("volume", 1.0)

_tts_lock    = threading.Lock()
tts_enabled  = True   # toggled by 🔊 button in GUI

def speak(text):
    """Speak text in a background thread (non-blocking)."""
    if not tts_enabled:
        return
    def _run():
        with _tts_lock:
            # strip markdown symbols that sound odd when spoken
            clean = text.replace("*","").replace("#","").replace("`","")
            _tts_engine.say(clean)
            _tts_engine.runAndWait()
    threading.Thread(target=_run, daemon=True).start()

# ── STT ────────────────────────────────────────────────────────────────────
_recognizer = sr.Recognizer()
_recognizer.pause_threshold = 1.0   # seconds of silence = end of speech

def listen_once(on_result, on_error, on_status):
    """
    Record one utterance from the microphone and call:
      on_result(text)  — when recognised successfully
      on_error(msg)    — on failure
      on_status(msg)   — for live status updates (called from worker thread)
    Always runs in a background thread; never blocks the GUI.
    """
    def _run():
        try:
            on_status("🎤 Listening…")
            with sr.Microphone() as source:
                _recognizer.adjust_for_ambient_noise(source, duration=0.4)
                audio = _recognizer.listen(source, timeout=8, phrase_time_limit=15)
            on_status("⏳ Recognising…")
            text = _recognizer.recognize_google(audio)
            on_result(text)
        except sr.WaitTimeoutError:
            on_error("No speech detected — try again.")
        except sr.UnknownValueError:
            on_error("Couldn't understand — please speak clearly.")
        except sr.RequestError as e:
            on_error(f"Speech service error: {e}")
        except Exception as e:
            on_error(f"Mic error: {e}")
    threading.Thread(target=_run, daemon=True).start()


## 📸 10 · Camera & Image Features

- **🖼️ Image Upload** — send any photo to Bemo and she'll analyse it with Gemini Vision  
- **📷 Live Camera** — capture a frame straight from your webcam and ask Bemo about it  
- Thumbnails appear inline in the chat bubble

In [13]:
import cv2
import base64
import tempfile
import os
from PIL import Image as PILImage
import google.generativeai as genai

_vision_model = genai.GenerativeModel("gemini-2.5-flash")

# ── Gemini Vision ──────────────────────────────────────────────────────────
def analyze_image(image_path, question="Describe this image in detail."):
    """Send an image to Gemini Vision and return the response text."""
    try:
        img = PILImage.open(image_path)
        prompt = (
            "You are Bemo 🤖 — a warm, witty AI assistant.\n"
            f"The user sent an image. Their question/context: {question}\n"
            "Analyse the image and respond naturally and helpfully."
        )
        res = _vision_model.generate_content([prompt, img])
        return res.text
    except Exception as e:
        return f"Sorry, I couldn't analyse the image: {e}"

# ── Camera capture ─────────────────────────────────────────────────────────
def capture_from_camera(on_captured, on_error):
    """
    Opens a live webcam preview in a Toplevel window.
    on_captured(image_path) — called with path of saved PNG
    on_error(msg)           — called on failure
    Always non-blocking; runs the preview loop inside tkinter's event loop.
    """
    import tkinter as tk
    from PIL import ImageTk

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        on_error("No camera found.")
        return

    win = tk.Toplevel()
    win.title("📷  Bemo Camera")
    win.configure(bg="#1e1e2e")
    win.resizable(False, False)

    lbl = tk.Label(win, bg="#1e1e2e")
    lbl.pack(padx=10, pady=10)

    btn_frame = tk.Frame(win, bg="#1e1e2e")
    btn_frame.pack(pady=(0, 10))

    _current_frame = [None]   # mutable container for closure

    def update_frame():
        ret, frame = cap.read()
        if ret:
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img   = PILImage.fromarray(frame_rgb)
            pil_img.thumbnail((640, 480))
            tk_img = ImageTk.PhotoImage(pil_img)
            lbl.configure(image=tk_img)
            lbl.image = tk_img          # keep reference
            _current_frame[0] = frame
        if win.winfo_exists():
            win.after(30, update_frame)

    def do_capture():
        if _current_frame[0] is None:
            return
        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        cv2.imwrite(tmp.name, _current_frame[0])
        cap.release()
        win.destroy()
        on_captured(tmp.name)

    def on_close():
        cap.release()
        win.destroy()

    tk.Button(btn_frame, text="📸  Capture", font=("Segoe UI", 11, "bold"),
              bg="#89b4fa", fg="#1e1e2e", relief="flat", padx=20, pady=6,
              cursor="hand2", command=do_capture).pack(side=tk.LEFT, padx=6)

    tk.Button(btn_frame, text="✕  Cancel", font=("Segoe UI", 11),
              bg="#313244", fg="#cdd6f4", relief="flat", padx=16, pady=6,
              cursor="hand2", command=on_close).pack(side=tk.LEFT, padx=6)

    win.protocol("WM_DELETE_WINDOW", on_close)
    update_frame()


In [14]:
import warnings
warnings.filterwarnings("ignore", message="datetime.datetime.utcnow() is deprecated")
warnings.filterwarnings("ignore", message="This package (`duckduckgo_search`) has been renamed")

In [15]:
import os
import tkinter as tk
from tkinter import filedialog

# ── Colour palette ──────────────────────────────────────────────────────────
BG          = "#1e1e2e"
CHAT_BG     = "#181825"
INPUT_BG    = "#313244"
BTN_BG      = "#89b4fa"
BTN_FG      = "#1e1e2e"
STATUS_CLR  = "#6c7086"
TEXT_CLR    = "#cdd6f4"
FONT_UI     = ("Segoe UI", 10)
FONT_CHAT   = ("Segoe UI", 11)

# bubble colours
USER_BUBBLE = "#005c4b"   # WhatsApp dark green
USER_TEXT   = "#e9edef"
BEMO_BUBBLE = "#202c33"   # WhatsApp dark grey
BEMO_TEXT   = "#e9edef"
BEMO_NAME   = "#53bdeb"   # light blue name label

# calculator colours
CALC_BG     = "#1a1a2e"
CALC_DISP   = "#16213e"
CALC_NUM    = "#0f3460"
CALC_OP     = "#e94560"
CALC_EQ     = "#533483"
CALC_CLR    = "#c84b31"
CALC_TXT    = "#eaeaea"

# ── WhatsApp-style scrollable chat ───────────────────────────────────────────
def make_chat_area(parent):
    container = tk.Frame(parent, bg=CHAT_BG)
    container.pack(fill=tk.BOTH, expand=True)

    canvas = tk.Canvas(container, bg=CHAT_BG, highlightthickness=0)
    scrollbar = tk.Scrollbar(container, orient="vertical", command=canvas.yview)
    canvas.configure(yscrollcommand=scrollbar.set)

    scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
    canvas.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

    inner = tk.Frame(canvas, bg=CHAT_BG)
    canvas_window = canvas.create_window((0, 0), window=inner, anchor="nw")

    def on_resize(event):
        canvas.itemconfig(canvas_window, width=event.width)
    canvas.bind("<Configure>", on_resize)

    def on_frame_change(event):
        canvas.configure(scrollregion=canvas.bbox("all"))
        canvas.yview_moveto(1.0)
    inner.bind("<Configure>", on_frame_change)

    # mouse wheel scroll
    def on_wheel(event):
        canvas.yview_scroll(int(-1 * (event.delta / 120)), "units")
    canvas.bind_all("<MouseWheel>", on_wheel)

    return inner, canvas

import tkinter.font as tkfont

def copy_text(text):
    root.clipboard_clear()
    root.clipboard_append(text)
    root.update()

def _rounded_rect(cv, x1, y1, x2, y2, r, **kw):
    """Draw a smooth rounded rectangle on a Canvas."""
    pts = [
        x1+r, y1,   x2-r, y1,
        x2,   y1,   x2,   y1+r,
        x2,   y2-r, x2,   y2,
        x2-r, y2,   x1+r, y2,
        x1,   y2,   x1,   y2-r,
        x1,   y1+r, x1,   y1,
    ]
    return cv.create_polygon(pts, smooth=True, **kw)

def add_bubble(inner, canvas, text, side):
    WRAP   = 460
    PX, PY = 14, 10
    R      = 18     # corner radius

    if side == "right":
        fill_clr    = USER_BUBBLE
        text_clr    = USER_TEXT
        pack_side   = tk.RIGHT
        margin      = (80, 8)
    else:
        fill_clr    = BEMO_BUBBLE
        text_clr    = BEMO_TEXT
        pack_side   = tk.LEFT
        margin      = (8, 80)

    # ── measure text to size the canvas ───────────────────────────────────
    fnt  = tkfont.Font(family="Segoe UI", size=11)
    lh   = fnt.metrics("linespace")

    wrapped_lines = []
    for para in text.split("\n"):
        words = para.split() or [""]
        cur   = words[0]
        for w in words[1:]:
            if fnt.measure(cur + " " + w) <= WRAP:
                cur += " " + w
            else:
                wrapped_lines.append(cur)
                cur = w
        wrapped_lines.append(cur)

    tw = min(max((fnt.measure(l) for l in wrapped_lines), default=60), WRAP)
    th = lh * len(wrapped_lines)

    cw = tw + PX * 2
    ch = th + PY * 2

    row = tk.Frame(inner, bg=CHAT_BG, pady=4)
    row.pack(fill=tk.X, padx=10)

    cv = tk.Canvas(row, width=cw, height=ch,
                   bg=CHAT_BG, highlightthickness=0, cursor="hand2")
    cv.pack(side=pack_side, padx=margin)

    rect_id = _rounded_rect(cv, 1, 1, cw-1, ch-1, R,
                             fill=fill_clr, outline=fill_clr)
    cv.create_text(PX, PY, text=text, anchor="nw",
                   fill=text_clr, font=FONT_CHAT, width=WRAP)

    # ── flash on copy ──────────────────────────────────────────────────────
    def flash():
        cv.itemconfig(rect_id, fill="#45475a", outline="#45475a")
        cv.after(180, lambda: cv.itemconfig(rect_id,
                                            fill=fill_clr, outline=fill_clr))

    def do_copy(t=text):
        copy_text(t)
        flash()

    # ── context menu ──────────────────────────────────────────────────────
    menu = tk.Menu(root, tearoff=0, bg=INPUT_BG, fg=TEXT_CLR,
                   activebackground=BTN_BG, activeforeground=BTN_FG,
                   relief="flat", bd=0)
    menu.add_command(label="📋  Copy", command=do_copy)

    cv.bind("<Button-3>",        lambda e: menu.tk_popup(e.x_root, e.y_root))
    cv.bind("<Double-Button-1>", lambda e: do_copy())

    canvas.update_idletasks()
    canvas.yview_moveto(1.0)

# ── Helpers ──────────────────────────────────────────────────────────────────
def set_status(msg):
    count = memory_count()
    status_var.set(f"{msg}   |   💬 {count} message(s) in memory")
    root.update_idletasks()

def clear_chat():
    memory.clear()
    for widget in chat_inner.winfo_children():
        widget.destroy()
    add_bubble(chat_inner, chat_canvas, "Hello! Ask me anything 🤖", "left")
    set_status("Chat cleared.")

# ── Send (threaded — UI never freezes) ───────────────────────────────────────
def send_message(event=None):
    user_text = user_input.get().strip()
    if not user_text:
        return
    add_bubble(chat_inner, chat_canvas, user_text, "right")
    user_input.delete(0, tk.END)
    send_btn.config(state="disabled")
    upload_btn.config(state="disabled")
    set_status("Bemo is thinking…")

    def run():
        try:
            reply = agent(user_text)
        except Exception as exc:
            reply = f"Sorry, an error happened: {exc}"
        root.after(0, lambda: finish_send(reply))

    def finish_send(reply):
        add_bubble(chat_inner, chat_canvas, reply, "left")
        speak(reply)                         # 🔊 TTS
        send_btn.config(state="normal")
        upload_btn.config(state="normal")
        set_status("Ready")
        user_input.focus_set()

    threading.Thread(target=run, daemon=True).start()

# ── File Upload (threaded) ────────────────────────────────────────────────────
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}

def _show_image_thumbnail(inner, canvas, img_path, label_text):
    """Show a small image thumbnail inside a chat bubble frame."""
    from PIL import ImageTk
    try:
        pil = PILImage.open(img_path)
        pil.thumbnail((260, 200))
        tk_img = ImageTk.PhotoImage(pil)
        row = tk.Frame(inner, bg=CHAT_BG, pady=4)
        row.pack(fill=tk.X, padx=10)
        lbl = tk.Label(row, image=tk_img, bg=USER_BUBBLE,
                       cursor="hand2", relief="flat", bd=2)
        lbl.image = tk_img          # keep reference
        lbl.pack(side=tk.RIGHT, padx=(80, 8))
        tk.Label(row, text=label_text, font=("Segoe UI", 9),
                 bg=CHAT_BG, fg=STATUS_CLR).pack(side=tk.RIGHT, anchor="s")
        canvas.update_idletasks()
        canvas.yview_moveto(1.0)
    except Exception:
        add_bubble(inner, canvas, label_text, "right")

def _handle_image_path(img_path, filename, question="What do you see in this image?"):
    """Shared logic: show thumbnail, call vision, show reply."""
    _show_image_thumbnail(chat_inner, chat_canvas, img_path, f"🖼️ {filename}")
    set_status(f"🔍 Bemo is analysing {filename}…")
    upload_btn.config(state="disabled")
    send_btn.config(state="disabled")

    def run():
        try:
            reply = analyze_image(img_path, question)
        except Exception as exc:
            reply = f"Error analysing image: {exc}"
        root.after(0, lambda: _finish_image(filename, reply, img_path))

    def _finish_image(fname, reply, path):
        add_bubble(chat_inner, chat_canvas, f"🖼️ About \'{fname}\':\n{reply}", "left")
        speak(reply)
        save_memory(f"[Image: {fname}]", reply)
        upload_btn.config(state="normal")
        send_btn.config(state="normal")
        set_status("Ready")

    threading.Thread(target=run, daemon=True).start()

def upload_file():
    path = filedialog.askopenfilename(
        title="Choose a file or image",
        filetypes=[
            ("Images",    "*.jpg *.jpeg *.png *.bmp *.gif *.webp"),
            ("Text files","*.txt *.md *.py *.csv *.json *.html *.xml"),
            ("All files", "*.*")
        ]
    )
    if not path:
        return
    filename = path.split("/")[-1].split("\\")[-1]
    ext = os.path.splitext(filename)[1].lower()

    if ext in IMAGE_EXTS:
        _handle_image_path(path, filename)
        return

    # ── text file path (original logic) ───────────────────────────────────
    add_bubble(chat_inner, chat_canvas, f"📄 Uploaded \'{filename}\'", "right")
    set_status(f"Summarizing {filename}…")
    upload_btn.config(state="disabled")
    send_btn.config(state="disabled")

    def run():
        try:
            summary = summarize_file(path)
        except Exception as exc:
            summary = f"Error reading file: {exc}"
        root.after(0, lambda: finish_upload(filename, summary))

    def finish_upload(fname, summary):
        add_bubble(chat_inner, chat_canvas, f"📄 Summary of \'{fname}\':\n{summary}", "left")
        save_memory(f"Summarize {fname}", summary)
        upload_btn.config(state="normal")
        send_btn.config(state="normal")
        set_status("Ready")

    threading.Thread(target=run, daemon=True).start()

# ── Calculator popup ──────────────────────────────────────────────────────────
def open_calculator():
    win = tk.Toplevel(root)
    win.title("Calculator")
    win.resizable(False, False)
    win.configure(bg=CALC_BG)

    expr = tk.StringVar(value="")

    display = tk.Entry(
        win, textvariable=expr,
        font=("Segoe UI", 18, "bold"),
        bg=CALC_DISP, fg=CALC_TXT,
        insertbackground=CALC_TXT,
        relief="flat", justify="right", bd=14
    )
    display.grid(row=0, column=0, columnspan=4, sticky="ew", padx=10, pady=10)

    def press(val):
        if val == "=":
            try:
                expr.set(str(eval(expr.get())))
            except:
                expr.set("Error")
        elif val == "C":
            expr.set("")
        elif val == "⌫":
            expr.set(expr.get()[:-1])
        else:
            expr.set(expr.get() + str(val))

    buttons = [
        ["C",  "⌫", "%",  "/"],
        ["7",  "8", "9",  "*"],
        ["4",  "5", "6",  "-"],
        ["1",  "2", "3",  "+"],
        ["00", "0", ".",  "="],
    ]

    for r, row in enumerate(buttons, start=1):
        for c, val in enumerate(row):
            if val == "=":
                bg, fg = CALC_EQ,  CALC_TXT
            elif val in ("C", "⌫"):
                bg, fg = CALC_CLR, CALC_TXT
            elif val in ("/", "*", "-", "+", "%"):
                bg, fg = CALC_OP,  CALC_TXT
            else:
                bg, fg = CALC_NUM, CALC_TXT
            tk.Button(
                win, text=val,
                font=("Segoe UI", 13, "bold"),
                bg=bg, fg=fg,
                activebackground=fg, activeforeground=bg,
                relief="flat", width=4, height=2,
                cursor="hand2",
                command=lambda v=val: press(v)
            ).grid(row=r, column=c, padx=3, pady=3)

# ── Window ────────────────────────────────────────────────────────────────────
# ── Voice state ───────────────────────────────────────────────────────────
tts_enabled = True   # will be overwritten by toggle button

root = tk.Tk()
root.title("Bemo — AI Chatbot")
root.geometry("820x640")
root.resizable(True, True)
root.configure(bg=BG)

# Header
header = tk.Frame(root, bg=BG, pady=8)
header.pack(fill=tk.X, padx=16)
tk.Label(header, text="🤖  Bemo", font=("Segoe UI", 16, "bold"),
         bg=BG, fg=BTN_BG).pack(side=tk.LEFT)
tk.Label(header, text="Powered by Gemini 2.5 Flash", font=FONT_UI,
         bg=BG, fg=STATUS_CLR).pack(side=tk.LEFT, padx=10)
tk.Button(header, text="🧮 Calc", font=FONT_UI, bg=INPUT_BG, fg=CALC_OP,
          relief="flat", padx=10, cursor="hand2",
          command=open_calculator).pack(side=tk.RIGHT, padx=(6, 0))

# 🔊 TTS toggle button
_tts_btn_var = tk.StringVar(value="🔊 ON")
def toggle_tts():
    global tts_enabled
    tts_enabled = not tts_enabled
    _tts_btn_var.set("🔊 ON" if tts_enabled else "🔇 OFF")
tk.Button(header, textvariable=_tts_btn_var, font=FONT_UI, bg=INPUT_BG,
          fg=BTN_BG, relief="flat", padx=10, cursor="hand2",
          command=toggle_tts).pack(side=tk.RIGHT, padx=(6, 0))
tk.Button(header, text="Clear", font=FONT_UI, bg=INPUT_BG, fg=TEXT_CLR,
          relief="flat", padx=10, cursor="hand2",
          command=clear_chat).pack(side=tk.RIGHT)

# Divider
tk.Frame(root, bg=INPUT_BG, height=1).pack(fill=tk.X)

# Chat area (WhatsApp-style)
chat_inner, chat_canvas = make_chat_area(root)

# Divider
tk.Frame(root, bg=INPUT_BG, height=1).pack(fill=tk.X)

# Input row
input_frame = tk.Frame(root, bg=BG, pady=10)
input_frame.pack(fill=tk.X, padx=14)

upload_btn = tk.Button(
    input_frame, text="📄", font=("Segoe UI", 13),
    bg=INPUT_BG, fg=BTN_BG,
    relief="flat", padx=8, pady=4,
    cursor="hand2", command=upload_file
)
upload_btn.pack(side=tk.LEFT, padx=(0, 8))

# 🎤 Microphone button
def start_listening():
    mic_btn.config(state="disabled")
    send_btn.config(state="disabled")

    def on_result(text):
        def _insert():
            user_input.delete(0, tk.END)
            user_input.insert(0, text)
            mic_btn.config(state="normal")
            send_btn.config(state="normal")
            set_status("🎤 Got it — press Send or Enter")
        root.after(0, _insert)

    def on_error(msg):
        root.after(0, lambda: set_status(f"❌ {msg}"))
        root.after(0, lambda: mic_btn.config(state="normal"))
        root.after(0, lambda: send_btn.config(state="normal"))

    def on_status(msg):
        root.after(0, lambda: set_status(msg))

    listen_once(on_result, on_error, on_status)

mic_btn = tk.Button(
    input_frame, text="🎤", font=("Segoe UI", 13),
    bg=INPUT_BG, fg="#f38ba8",
    relief="flat", padx=8, pady=4,
    cursor="hand2", command=start_listening
)
mic_btn.pack(side=tk.LEFT, padx=(0, 8))

# 📷 Camera button
def open_camera():
    cam_btn.config(state="disabled")
    send_btn.config(state="disabled")

    def on_captured(img_path):
        fname = os.path.basename(img_path)
        q = user_input.get().strip() or "What do you see in this image?"
        user_input.delete(0, tk.END)
        root.after(0, lambda: _handle_image_path(img_path, "camera_capture.png", q))
        root.after(0, lambda: cam_btn.config(state="normal"))
        root.after(0, lambda: send_btn.config(state="normal"))

    def on_error(msg):
        root.after(0, lambda: set_status(f"❌ Camera: {msg}"))
        root.after(0, lambda: cam_btn.config(state="normal"))
        root.after(0, lambda: send_btn.config(state="normal"))

    # capture_from_camera is non-blocking; it opens a Toplevel window
    root.after(0, lambda: capture_from_camera(on_captured, on_error))

cam_btn = tk.Button(
    input_frame, text="📷", font=("Segoe UI", 13),
    bg=INPUT_BG, fg="#a6e3a1",
    relief="flat", padx=8, pady=4,
    cursor="hand2", command=open_camera
)
cam_btn.pack(side=tk.LEFT, padx=(0, 8))

user_input = tk.Entry(
    input_frame, font=FONT_CHAT,
    bg=INPUT_BG, fg=TEXT_CLR,
    insertbackground=TEXT_CLR,
    relief="flat", bd=6
)
user_input.pack(side=tk.LEFT, fill=tk.X, expand=True, ipady=6)
user_input.bind("<Return>", send_message)

send_btn = tk.Button(
    input_frame, text="Send ➤",
    font=("Segoe UI", 10, "bold"),
    bg=BTN_BG, fg=BTN_FG,
    relief="flat", padx=18, pady=6,
    cursor="hand2", command=send_message
)
send_btn.pack(side=tk.RIGHT, padx=(10, 0))

# Status bar
status_var = tk.StringVar(value="Ready")
tk.Label(root, textvariable=status_var, font=("Segoe UI", 9),
         bg=BG, fg=STATUS_CLR, anchor="w"
         ).pack(fill=tk.X, padx=16, pady=(0, 6))

# First message
add_bubble(chat_inner, chat_canvas, "Hello! Ask me anything 🤖", "left")
user_input.focus_set()
root.mainloop()
